In [4]:
import sqlite3
import pandas as pd
import joblib
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

def preprocess_text(text):
    """Consistent text cleaning for training and inference."""
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

def train():
    # 1. Load Data
    conn = sqlite3.connect('Classification.db')
    query = "SELECT * FROM fake_news_classification"
    df = pd.read_sql_query(query, conn)
    conn.close()

       # 2. Split Data per requirements - add .copy() to the end
    train_df = df.iloc[:55031].copy()
    eval_df = df.iloc[55031:55031+15723].copy()
    
    # 3. Preprocess (The error will now disappear)
    train_df['clean_text'] = train_df['text'].apply(preprocess_text)
    eval_df['clean_text'] = eval_df['text'].apply(preprocess_text)

    # 4. Feature Extraction (TF-IDF)
    tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
    X_train = tfidf.fit_transform(train_df['clean_text'])
    y_train = train_df['label'] # Assuming column name is 'label'
    
    X_eval = tfidf.transform(eval_df['clean_text'])
    y_eval = eval_df['label']

    # 5. Model Selection: Random Forest
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # 6. Evaluation
    preds = model.predict(X_eval)
    print("Evaluation Results:")
    print(classification_report(y_eval, preds))

    # 7. Save Model and Vectorizer
    joblib.dump(model, 'fake_news_model.pkl')
    joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
    print("Model and Vectorizer saved.")

if __name__ == "__main__":
    train()


Evaluation Results:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95      8730
           1       0.95      0.93      0.94      6993

    accuracy                           0.95     15723
   macro avg       0.95      0.94      0.94     15723
weighted avg       0.95      0.95      0.95     15723

Model and Vectorizer saved.
